# Whisper Small Telephony Fine-Tuning

Run this notebook in Google Colab after copying the staged dataset to:

`/content/drive/MyDrive/carecaller/`

Expected layout:

```text
carecaller/
  telephony_manifest.csv
  telephony_manifest_summary.json
  audio/
    *.wav
```


In [1]:
!pip -q install -U transformers datasets accelerate evaluate jiwer librosa soundfile torchaudio sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [16]:
import json, csv
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/carecaller")
src = BASE_DIR / "clips_rich_noise.jsonl"
out = BASE_DIR / "telephony_manifest.csv"

rows = []
for line in src.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    rows.append({
        "audio_path": str((BASE_DIR / row["audio_telephony_path"]).resolve()),
        "text": row["text"],
        "split": row["split"],
    })

with out.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["audio_path", "text", "split"])
    writer.writeheader()
    writer.writerows(rows)

print("wrote", out)
print(rows[0]["audio_path"])

wrote /content/drive/MyDrive/carecaller/telephony_manifest.csv
/content/drive/MyDrive/carecaller/telephony/clip_0001_clean.wav


In [4]:
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/carecaller')
MANIFEST = BASE_DIR / 'telephony_manifest.csv'
OUTPUT_DIR = BASE_DIR / 'whisper_small_telephony'
SAMPLE_RATE = 16000
MODEL_ID = 'openai/whisper-small'
LANGUAGE = 'english'
TASK = 'transcribe'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('manifest:', MANIFEST)
print('output:', OUTPUT_DIR)
if not MANIFEST.exists():
    raise FileNotFoundError(f'Manifest not found: {MANIFEST}')


manifest: /content/drive/MyDrive/carecaller/telephony_manifest.csv
output: /content/drive/MyDrive/carecaller/whisper_small_telephony


In [5]:
from datasets import Audio, DatasetDict, load_dataset

data = load_dataset("csv", data_files={"train": str(MANIFEST)})["train"]

required_columns = {'audio_path', 'text', 'split'}
missing = required_columns - set(data.column_names)
if missing:
    raise ValueError(f'Manifest missing required columns: {sorted(missing)}')

train_ds = data.filter(lambda x: x['split'] == 'train')
val_ds = data.filter(lambda x: x['split'] == 'val')
test_ds = data.filter(lambda x: x['split'] == 'test')

ds = DatasetDict({'train': train_ds, 'validation': val_ds, 'test': test_ds})
ds = ds.cast_column('audio_path', Audio(sampling_rate=SAMPLE_RATE))
ds


Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/320 [00:00<?, ? examples/s]

Filter:   0%|          | 0/320 [00:00<?, ? examples/s]

Filter:   0%|          | 0/320 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio_path', 'text', 'split'],
        num_rows: 192
    })
    validation: Dataset({
        features: ['audio_path', 'text', 'split'],
        num_rows: 64
    })
    test: Dataset({
        features: ['audio_path', 'text', 'split'],
        num_rows: 64
    })
})

In [6]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.config.suppress_tokens = []
model.config.use_cache = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

device: cuda


In [7]:
import json, csv
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/carecaller")
src = BASE_DIR / "clips_rich_noise.jsonl"
out = BASE_DIR / "telephony_manifest.csv"

rows = []
for line in src.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    rows.append({
        "audio_path": str(BASE_DIR / row["audio_telephony_path"]),
        "text": row["text"],
        "split": row["split"],
    })

with out.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["audio_path", "text", "split"])
    writer.writeheader()
    writer.writerows(rows)

print(out)


/content/drive/MyDrive/carecaller/telephony_manifest.csv


In [8]:
def prepare_batch(batch):
    audio = batch['audio_path']
    batch['input_features'] = processor.feature_extractor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['text']).input_ids
    batch['input_length'] = len(audio['array']) / audio['sampling_rate']
    return batch

ds = ds.map(
    prepare_batch,
    remove_columns=ds['train'].column_names,
    num_proc=1,
)

MAX_AUDIO_SECONDS = 30.0
ds['train'] = ds['train'].filter(lambda x: x['input_length'] < MAX_AUDIO_SECONDS)
ds['validation'] = ds['validation'].filter(lambda x: x['input_length'] < MAX_AUDIO_SECONDS)
ds['test'] = ds['test'].filter(lambda x: x['input_length'] < MAX_AUDIO_SECONDS)

ds


Map (num_proc=1):   0%|          | 0/192 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/64 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/64 [00:00<?, ? examples/s]

Filter:   0%|          | 0/192 [00:00<?, ? examples/s]

Filter:   0%|          | 0/64 [00:00<?, ? examples/s]

Filter:   0%|          | 0/64 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels', 'input_length'],
        num_rows: 192
    })
    validation: Dataset({
        features: ['input_features', 'labels', 'input_length'],
        num_rows: 64
    })
    test: Dataset({
        features: ['input_features', 'labels', 'input_length'],
        num_rows: 64
    })
})

In [9]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': feature['input_features']} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': feature['labels']} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


In [10]:
import evaluate

wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer, 'cer': cer}


In [13]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=100,
    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),
    eval_strategy='steps',
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    report_to='none',
)


In [14]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()


Step,Training Loss,Validation Loss,Wer,Cer
100,0.286349,0.194316,2.466907,1.319350


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=100, training_loss=1.0646862268447876, metrics={'train_runtime': 507.3595, 'train_samples_per_second': 3.154, 'train_steps_per_second': 0.197, 'total_flos': 4.61736640512e+17, 'train_loss': 1.0646862268447876, 'epoch': 8.333333333333334})

In [23]:
# After training is done
metrics = trainer.predict(ds['test']).metrics
print(metrics)

final_dir = OUTPUT_DIR / 'final'
trainer.save_model(str(final_dir))
processor.save_pretrained(str(final_dir))

with (final_dir / 'metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)

{'test_loss': 0.4640473425388336, 'test_model_preparation_time': 0.0053, 'test_wer': 22.26233453670277, 'test_cer': 17.534957149300855, 'test_runtime': 35.4293, 'test_samples_per_second': 1.806, 'test_steps_per_second': 0.226}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
import librosa
import torch
import os
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from pathlib import Path

# Using relative path from root to avoid Hub validation triggers
INFER_DIR = "drive/MyDrive/carecaller/whisper_small_telephony/final"
SAMPLE_AUDIO = "/content/drive/MyDrive/carecaller/sample.wav"

print(f"Loading model from local directory: {INFER_DIR}")

if not os.path.isdir(INFER_DIR):
    print(f"Error: Directory not found at {INFER_DIR}")
else:
    # Load processor and model
    processor = WhisperProcessor.from_pretrained(INFER_DIR, local_files_only=True)
    model = WhisperForConditionalGeneration.from_pretrained(INFER_DIR, local_files_only=True).to(device)

    def transcribe_file(path: str) -> str:
        audio, _ = librosa.load(path, sr=16000, mono=True)
        inputs = processor(audio, sampling_rate=16000, return_tensors='pt')
        input_features = inputs.input_features.to(device)

        predicted_ids = model.generate(input_features)
        return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    if Path(SAMPLE_AUDIO).exists():
        print("Transcription result:")
        print(transcribe_file(SAMPLE_AUDIO))
    else:
        print(f"File not found: {SAMPLE_AUDIO}")

Loading model from local directory: drive/MyDrive/carecaller/whisper_small_telephony/final


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Transcription result:
My child took acetaminophen at 7 PM and then ibuprofen at 8 PM, and I want to make sure that spacing was actually safe. She still has a low fever, and I do not want to give the next dose too early if I wrote the schedule down wrong. I wrote the times on the box flap in a hurry, so I am worried I may have mixed up which medicine came first. If the schedule needs to be reset tonight, I would rather fix it now than keep guessing.


In [25]:
import shutil
from google.colab import files
from pathlib import Path

# Define the directory to zip and the output filename
model_path = '/content/drive/MyDrive/carecaller/whisper_small_telephony/final'
zip_filename = '/content/whisper_small_telephony_final.zip'

if Path(model_path).exists():
    print(f'Zipping model directory: {model_path}')
    shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', model_path)
    print(f'Zip created at: {zip_filename}')

    print('Starting download to your computer...')
    files.download(zip_filename)
else:
    print(f'Error: Model directory not found at {model_path}. Please check if the training and saving cells finished successfully.')

Zipping model directory: /content/drive/MyDrive/carecaller/whisper_small_telephony/final
Zip created at: /content/whisper_small_telephony_final.zip
Starting download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>